In [1]:
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline, griddata
from scipy.stats import norm
from skew.utils import *
import nest_asyncio, asyncio
nest_asyncio.apply()


# ===============================
# CONFIG
# ===============================
DATA_SOURCE   = "yfinance"   # "yfinance" or "ibkr"
TICKERS        = ['NVDA', 'QQQ', 'MSFT', 'TSM']        # underlying ticker
MAX_EXPIRIES  = 50           # how many expiries to fetch
MIN_EXPIRY_DAYS = 10            # ignore expiries below this
USE_CALLS     = True         # use calls (could also use puts)
SMOOTH_S      = 1e-1         # spline smoothing parameter per expiry
EVAL_K_GRID   = 100           # how many strikes to evaluate per expiry

In [2]:
from skew.zero_curve import RiskFreeCurveFactory
from datetime import date, timedelta

# Create a piecewise linear zero curve for USD (fetches latest from Fed)
curve_usd = RiskFreeCurveFactory.create("USD")

print("Valuation date:", curve_usd.valuation_date)
print("SOFR rate:", round(curve_usd.rate * 100, 3), "%")
print("1Y DF:", round(curve_usd.discount_factor(curve_usd.valuation_date + timedelta(days=360)), 5))



Valuation date: 2026-04-15
SOFR rate: 3.91 %
1Y DF: 0.96217


In [3]:
import pandas as pd
from datetime import date
from skew.zero_curve import PiecewiseLinearZeroCurve

# Usage
pillars = [
    ("1D", 0.04),
    ("1M", 0.0401),
    ("3M", 0.0389),
    ("1Y", 0.0354),
    ("2Y", 0.0333),
    ("5Y", 0.0334),
]
curve = PiecewiseLinearZeroCurve(
    valuation_date=date.today(),
    pillars=pillars,
    day_count="ACT/365F"
)

print("Valuation date:", curve.valuation_date)
print("1D rate:", round(curve.zero_rate(curve.valuation_date + timedelta(days=1))*100, 3), "%")
print("1Y rate:", round(curve.zero_rate(curve.valuation_date + timedelta(days=65))*100, 3), "%")
print("1Y DF:", round(curve.discount_factor(curve.valuation_date + timedelta(days=65)), 5))

# Now you can get discount factors, forwards etc.


Valuation date: 2026-04-15
1D rate: 4.0 %
1Y rate: 3.94 %
1Y DF: 0.99301


In [5]:

from attr import ib
from ib_insync import *
import logging

# ===============================
# DATA: IBKR option surface
# (requires IB Gateway/TWS running)
# ===============================

async def fetch_surface_ibkr_nb(ticker: str, max_expiries=10, batch_size=10):

    util.startLoop()
    ib = IB()
    util.logToConsole(level=logging.CRITICAL)   # or CRITICAL for total silence

    # Connect asynchronously (safe in notebooks)
    if not ib.isConnected():
        await ib.connectAsync("127.0.0.1", 7497, clientId=1)
        print('Connected:', ib.isConnected())

    ib.reqMarketDataType(4)
    ib.sleep(2.0)
    stk = Stock(ticker, "SMART", "USD")
    ib.qualifyContracts(stk)
    mkt = ib.reqMktData(stk, "", False, False)
    ib.sleep(1.0) #await
    spot = float(mkt.last or mkt.close)

    chains = ib.reqSecDefOptParams(stk.symbol, "", stk.secType, stk.conId)
    chain = [c for c in chains if c.tradingClass == stk.symbol][0]
    #choose expiries between 1 week and ~18 months
    today = pd.Timestamp.today()
    min_date = today + pd.Timedelta(days=7)  # 1 week
    max_date = today + pd.Timedelta(days=400)  # ~18 months (365 + 182 days)
    expiries = sorted(
        pd.to_datetime(exp) for exp in chain.expirations 
        if min_date <= pd.to_datetime(exp) <= max_date
    )[:max_expiries]

    rows = [] #ight = [], ("C" if use_calls else "P")
    for exp in expiries:
        exp_str = exp.strftime("%Y%m%d")
        strikes = sorted(k for k in chain.strikes if k % 1 == 0) #[::2]
        contracts = [Option(symbol=ticker, 
                            lastTradeDateOrContractMonth=exp_str,
                            strike=k,
                            right=right, 
                            exchange="SMART",
                            currency="USD") for k in strikes for right in ("C", "P")]
        ib.qualifyContracts(*contracts)

        # ---- request streaming mkt data in small batches ----
        for i in range(0, len(contracts), batch_size):
            batch = contracts[i:i+batch_size]
            for c in batch:
                ib.reqMktData(c, "", False, False)  # streaming; Greeks will populate
            ib.sleep(2.0)

            # collect tickers for this batch
            for c in batch:
                tk = ib.ticker(c)  # current Ticker for this contract
                if not tk:
                    continue
                g = getattr(tk, "modelGreeks", None)
                delta = getattr(g, "delta", None)
                if not delta:
                    continue
                undPrice = getattr(g, "undPrice", None) 
                if((c.right == 'C' and (delta > 0.1 and delta < 0.9) and c.strike < undPrice) or 
                   (c.right == 'P' and (-delta > 0.1 and -delta < 0.9) and c.strike > undPrice)):
                    iv = getattr(g, "impliedVol", None) 
                    if iv and 0.0001 < iv < 3.0:
                        bid = tk.bid or np.nan
                        ask = tk.ask or np.nan
                        mid = np.nan
                        if np.isfinite(bid) and np.isfinite(ask) and bid > 0 and ask > 0:
                            mid = 0.5 * (bid + ask)
                        elif np.isfinite(bid) and bid > 0:
                            mid = bid
                        elif np.isfinite(ask) and ask > 0:
                            mid = ask
                        rows.append({
                            "contract": tk.contract.conId,
                            "strike": c.strike,
                            "type": c.right,
                            "expiry": pd.to_datetime(c.lastTradeDateOrContractMonth),
                            "delta": float(delta),
                            "underlying": float(undPrice),
                            "price": float(mid),
                            "iv": float(iv)
                        })
    
    ib.disconnect()
    df = pd.DataFrame(rows)
    df["days_to_expiry"] = (df["expiry"] - pd.Timestamp.today().normalize()).dt.days
    df = df[df["days_to_expiry"] > 1].copy()
    df["T"] = df["days_to_expiry"] / 365.0
    # Create a date string: YYYYMMDD
    today_str = today.strftime("%Y%m%d")
    filename = f"data/option_data_ibkr_{ticker}_{today_str}.csv"
    df.to_csv(filename, index=False)
    filename = f"data/option_data_ibkr_{ticker}_{today_str}.csv"
    df.to_csv(filename, index=False)

# ===============================
# DATA: yfinance option surface
# ===============================
def fetch_surface_yfinance(ticker, max_expiries=10, min_expiry_days=10):
    import yfinance as yf

    yf_t = yf.Ticker(ticker)
    spot = yf_t.history(period="1d")["Close"].iloc[-1]

    expiries = yf_t.options
    if not expiries:
        raise ValueError("No expiries returned by yfinance for this ticker.")
    expiries = expiries[2:max_expiries]

    frames = []
    for exp in expiries:
        chain = yf_t.option_chain(exp)
        for opt_type, tbl in [("call", chain.calls), ("put", chain.puts)]:
            # Standardize / clean
            tbl = tbl.rename(columns={"impliedVolatility": "iv"})
            tbl["expiry"] = pd.to_datetime(exp).tz_localize(None)
            tbl["isCall"] = (opt_type == "call")
            frames.append(tbl[["strike", "iv", "lastPrice", "bid", "ask", "expiry", "inTheMoney","isCall"]])
    df = pd.concat(frames, ignore_index=True)
    # Clean IVs
    df = df[df["iv"].between(0.01, 3.0)].copy()
    df["days_to_expiry"] = (df["expiry"] - pd.Timestamp.today().normalize()).dt.days
    df = df[df["days_to_expiry"] > 1]  # avoid same-day
    df["T"] = df["days_to_expiry"] / 365.0
    today = pd.Timestamp.today()
    today_str = today.strftime("%Y%m%d")
    df["expiry_date"] = df["expiry"].dt.date
    df["disc_factor"] = df["expiry_date"].apply(curve_usd.discount_factor)
    # crude dividend yield proxy (fallback to 0 if missing)
    info = yf_t.info or {}
    q_div = info.get("dividendYield", 0.0) or 0.0
    df["div_yield"] = float(q_div/100)
    df["forward"] = spot * np.exp(- q_div/100 * df["T"]) * df["disc_factor"]
    df["spot"] = spot


    filename = f"data/option_data_yf_{ticker}_{today_str}.csv"
    df.to_csv(filename, index=False)
    


In [6]:
if __name__ == "__main__":
    for TICKER in TICKERS:
        if DATA_SOURCE == "ibkr":
            spot, df_opts = asyncio.run(fetch_surface_ibkr_nb(
                TICKER, max_expiries=MAX_EXPIRIES, use_calls=USE_CALLS, batch_size=10))
        elif DATA_SOURCE == "yfinance":
            fetch_surface_yfinance(
                    TICKER, max_expiries=MAX_EXPIRIES, min_expiry_days=MIN_EXPIRY_DAYS)
        else:
            raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

        print(f"Fetched option data for {TICKER} from {DATA_SOURCE}:")
    #print(f"Spot price: {spot:.2f}")
    #print(f"Number of options: {len(df_opts)}")

Fetched option data for NVDA from yfinance:
Fetched option data for QQQ from yfinance:
Fetched option data for MSFT from yfinance:
Fetched option data for TSM from yfinance:
